# 03 — PriceOracle evaluation on NYISO test window

Compares GridFM / Chronos-2 / Seasonal-naïve forecasts on the 14-day test slice. With no GPU and no `chronos-forecasting`/`gridfm` packages installed, both fall through to the naïve baseline — the `source` label on each PriceForecast reports which code path produced the numbers.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from aerogrid.price_oracle import load_price_history, make_oracle
from aerogrid.config import NYISO_TEST_START, NYISO_TEST_END

prices = load_price_history()
print(f"loaded {len(prices):,} rows spanning {prices['timestamp'].min()} .. {prices['timestamp'].max()}")

# Train/test shapes
fig, ax = plt.subplots(figsize=(12,3))
ax.plot(prices["timestamp"], prices["lbmp"], lw=0.5, label="LBMP")
ax.axvline(NYISO_TEST_START, color="red", ls="--", label="train/test split")
ax.set_ylabel("LBMP ($/MWh)"); ax.legend(); ax.set_title("NYISO 15-min prices")
plt.tight_layout(); plt.show()

In [ ]:
# Rolling 24-h forecasts on each day of the test window.
from datetime import timedelta
realized = prices[(prices["timestamp"] >= NYISO_TEST_START) & (prices["timestamp"] < NYISO_TEST_END)]
impls = ["gridfm", "chronos", "naive"]
errors = {}
for impl in impls:
    oracle = make_oracle(impl)
    preds = []
    for day in pd.date_range(NYISO_TEST_START, NYISO_TEST_END - timedelta(days=1), freq="1D"):
        fc = oracle.get_15min_forecast(day, prices)
        preds.extend(fc.median)
    preds = np.array(preds[: len(realized)])
    mape = (np.abs(preds - realized["lbmp"].to_numpy()) / np.maximum(np.abs(realized["lbmp"].to_numpy()), 1)).mean() * 100
    errors[impl] = {"mape": mape, "source": fc.source}
    print(f"{impl:>10s}  source={fc.source:<30s} MAPE={mape:.1f}%")

# One-day closeup
day0 = NYISO_TEST_START
slice_real = realized[(realized["timestamp"] >= day0) & (realized["timestamp"] < day0 + timedelta(days=1))]
plt.figure(figsize=(12,4))
plt.plot(slice_real["timestamp"], slice_real["lbmp"], "k-", label="realized", lw=2)
for impl in impls:
    fc = make_oracle(impl).get_15min_forecast(day0, prices)
    t = pd.date_range(fc.slot_start, periods=len(fc.median), freq="15min", tz="UTC")
    plt.plot(t, fc.median, ls="--", label=f"{impl} ({fc.source})")
plt.ylabel("LBMP ($/MWh)"); plt.legend(); plt.title(f"Day 1 forecasts"); plt.tight_layout(); plt.show()